In [1]:
import os
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from typing import List, Optional
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import PromptTemplate
from supabase import create_client

In [2]:
load_dotenv()

True

In [3]:
class ExtractedJobRequirements(BaseModel):
    role: str = Field(description="The role a candidate has to perform in the job.")
    skills: List[str] = Field(description="The list of skills required for the job.")
    min_experience_years: int = Field(description="Minimum years of experience needed for the job. If not mentioned then consider it a fresher joba and keep the value as 0.", default = 0)
    max_salary_offered: Optional[int] = Field(description="Maximum amount of salary offered for the job in rupees per")
    location: Optional[str] = Field(description="The location of the job.")

In [4]:
prompt = PromptTemplate(
        template="""
        You are an experienced job description extractor.
        Your task is to extract the job requirements from the job description.
        You are expected to extract the following job requirements from the job description:
        - The role a candidate has to perform in the job.
        - The list of skills required for the job.
        - The minimum years of experience needed for the job.
        - The maximum amount of salary offered for the job in rupees per annum.
        - The location of the job(if applicable).

        Here is the job description: {job_description}
        """,
        input_variables=["job_description"],
    )

In [5]:
llm_client = ChatOpenAI(model="gpt-4o-mini", openai_api_key=os.getenv("OPENAI_API_KEY"))

llm_with_tool = llm_client.with_structured_output(ExtractedJobRequirements)

chain = prompt | llm_with_tool

In [6]:
response = chain.invoke({"job_description": """
Senior AI/ML Engineer – Talent Intelligence Platform (Mumbai)

Company Overview
We are building an AI-powered recruitment intelligence platform that helps enterprises match candidates to roles using semantic search, embeddings, LLM-based resume analysis, and automated ranking pipelines. Our stack combines modern data engineering, vector databases, and generative AI to improve hiring efficiency at scale.

Role Overview

We are looking for a Senior AI/ML Engineer to design and build scalable machine learning systems for resume parsing, job-candidate matching, retrieval-augmented generation (RAG), and ranking pipelines. You will work closely with backend engineers, product teams, and data scientists to deploy production-grade AI systems.

This role requires strong experience with Python, LLM orchestration frameworks, vector databases, cloud infrastructure, and asynchronous processing pipelines.

Responsibilities
Build and maintain end-to-end ML pipelines for candidate-job matching.
Develop semantic search systems using embeddings and vector databases.
Design RAG pipelines using modern LLM frameworks.
Implement resume ingestion and parsing workflows for PDFs and DOCX files.
Optimize retrieval quality using hybrid search strategies (BM25 + vector similarity).
Create APIs and microservices for ML inference.
Work with async data ingestion pipelines handling thousands of resumes.
Fine-tune prompts and evaluation systems for extraction accuracy.
Collaborate with product teams to improve ranking relevance and explainability.
Monitor model performance, latency, and infrastructure costs.
Write clean, maintainable, and testable Python code.
Required Qualifications
4+ years of experience in machine learning or backend AI systems.
Strong proficiency in Python.
Experience with:
FastAPI or Flask
PostgreSQL
Vector databases such as pgvector, Pinecone, Weaviate, or Chroma
LangChain or LlamaIndex
OpenAI, Anthropic, or open-source LLMs
Understanding of:
Embeddings
RAG architectures
Semantic retrieval
Ranking systems
Prompt engineering
Experience working with cloud platforms such as AWS, GCP, or Azure.
Familiarity with Docker and CI/CD pipelines.
Strong debugging and system design skills.
Preferred Qualifications
Experience with Supabase and pgvector.
Knowledge of distributed systems and async programming.
Familiarity with resume parsing or HR-tech products.
Experience with Kafka, RabbitMQ, or event-driven architectures.
Understanding of evaluation metrics for retrieval and generation quality.
Contributions to open-source AI projects are a plus.
Tech Stack
Python
FastAPI
PostgreSQL + pgvector
LangChain
OpenAI APIs
Docker
AWS S3
Redis
Supabase
GitHub Actions
Sample Projects You May Work On
Building a hybrid retrieval engine for job matching.
Designing a scalable resume ingestion pipeline from OneDrive and S3.
Improving ranking quality using LLM-based reranking.
Developing automated candidate scoring systems.
Creating structured extraction pipelines from unstructured resumes.
Compensation & Benefits
Competitive salary
Remote-friendly work culture
Learning budget for AI/ML courses and conferences
Flexible working hours
Opportunity to work on cutting-edge GenAI systems
Application Instructions

Please submit:

Resume
GitHub profile
Links to relevant AI/ML projects
Brief explanation of a production ML system you have built

Candidates with hands-on project experience in RAG systems, vector search, or LLM applications will be prioritized.
"""})

In [7]:
print(response)

role='Senior AI/ML Engineer' skills=['Python', 'FastAPI', 'Flask', 'PostgreSQL', 'Vector databases (pgvector, Pinecone, Weaviate, Chroma)', 'LangChain', 'LlamaIndex', 'OpenAI', 'Anthropic', 'LLMs', 'Embeddings', 'RAG architectures', 'Semantic retrieval', 'Ranking systems', 'Prompt engineering', 'AWS', 'GCP', 'Azure', 'Docker', 'CI/CD pipelines'] min_experience_years=4 max_salary_offered=None location='Mumbai'


In [8]:
jd_str = f"""
Role: {response.role}
Skills: {", ".join(response.skills)}
"""

jd_str = jd_str.strip()

In [10]:
jd_str

'Role: Senior AI/ML Engineer\nSkills: Python, FastAPI, Flask, PostgreSQL, Vector databases (pgvector, Pinecone, Weaviate, Chroma), LangChain, LlamaIndex, OpenAI, Anthropic, LLMs, Embeddings, RAG architectures, Semantic retrieval, Ranking systems, Prompt engineering, AWS, GCP, Azure, Docker, CI/CD pipelines'

In [11]:
embedding_model = OpenAIEmbeddings(openai_api_key=os.getenv("OPENAI_API_KEY"))

jd_embeddings = embedding_model.embed_query(jd_str)

In [12]:
supabase = create_client(
    supabase_url=os.getenv("SUPABASE_URL"),
    supabase_key=os.getenv("SUPABASE_KEY"),
)

In [13]:
similar_candidates = (
    supabase.rpc(
        "match_candidates",
        {
            "query_embedding": jd_embeddings,
            "required_skills": response.skills,
            "min_experience_years": response.min_experience_years,
        }
    ).execute()
)

print(len(similar_candidates.data))

10


In [18]:
candidate_strings = []

for candidate in similar_candidates.data:
    candidate_str = f"""
    Name: {candidate['name']}
    Skills: {", ".join(candidate['skills'])}
    Experience: {candidate['years_experience']} years
    Last CTC: {candidate['last_ctc_inr']} rupees per annum
    Location: {candidate['current_location']}
    """

    candidate_strings.append(candidate_str)

all_candidate_str = "\n".join(candidate_strings)

print(all_candidate_str)    


    Name: Piyush Jain
    Skills: Python, Flask, PostgreSQL
    Experience: 4 years
    Last CTC: 1500000 rupees per annum
    Location: Jaipur
    

    Name: Amit Verma
    Skills: Python, Django, PostgreSQL
    Experience: 4 years
    Last CTC: 1200000 rupees per annum
    Location: Mumbai
    

    Name: Shalini Reddy
    Skills: Machine Learning, TensorFlow, Python
    Experience: 4 years
    Last CTC: 1800000 rupees per annum
    Location: Hyderabad
    

    Name: Siddharth Roy
    Skills: Python, Airflow, ETL
    Experience: 5 years
    Last CTC: 1600000 rupees per annum
    Location: Kolkata
    

    Name: Manoj Singh
    Skills: Python, ETL, Airflow
    Experience: 5 years
    Last CTC: 1800000 rupees per annum
    Location: Kolkata
    

    Name: Rashmi Kulkarni
    Skills: Python, Flask, SQLAlchemy
    Experience: 4 years
    Last CTC: 1400000 rupees per annum
    Location: Pune
    

    Name: Pooja Reddy
    Skills: Python, TensorFlow, NLP
    Experience: 4 years
    L

In [23]:
jd_str += f"\nMinimum experience years: {response.min_experience_years} years"

if response.max_salary_offered:
    jd_str += f"\nMaximum salary offered: {response.max_salary_offered} rupees per annum"

if response.location:
    jd_str += f"\nLocation: {response.location}"

In [24]:
jd_str

'Role: Senior AI/ML Engineer\nSkills: Python, FastAPI, Flask, PostgreSQL, Vector databases (pgvector, Pinecone, Weaviate, Chroma), LangChain, LlamaIndex, OpenAI, Anthropic, LLMs, Embeddings, RAG architectures, Semantic retrieval, Ranking systems, Prompt engineering, AWS, GCP, Azure, Docker, CI/CD pipelines\nLocation: Mumbai\nMinimum experience years: 4 years\nLocation: Mumbai'

In [19]:
class RankedCandidate(BaseModel):
    candidate_name: str = Field(description="The name of the candidate.")
    rationale: str = Field(description="The rationale behind the ranking.")

In [20]:
class RankedCandidatesResponse(BaseModel):
    ranked_candidates: list[RankedCandidate]

In [ ]:
prompt = PromptTemplate(
    template = """
    You are an expert technical recruiter.
    Your task is to rank the eligible candidates for the following job description.

    Here is the job description:
    {jd_str}

    Candidates:
    {all_candidates_str}

    Instructions:
    - Rank candidates from best to worst fit
    - Consider skills, experience, and relevance
    - Give a concise 1 line rationale for each candidate
    - Return structured list of ranked candidates.
    """,
    input_variables=["jd_str", "all_candidates_str"],
)

In [26]:
ranker_llm = llm_client.with_structured_output(RankedCandidatesResponse)

chain = prompt | ranker_llm

response = chain.invoke({"jd_str": jd_str, "all_candidates_str": all_candidate_str})

print(response.ranked_candidates)

[RankedCandidate(candidate_name='Amit Verma', rationale='Strong match with required skills (Python, PostgreSQL) and located in Mumbai. Experience of 4 years aligns well with the minimum requirement.'), RankedCandidate(candidate_name='Shivam Yadav', rationale='Relevant skills include Python and FastAPI, aligning with the job description. Experience of 4 years is adequate, although location is Noida.'), RankedCandidate(candidate_name='Pooja Reddy', rationale='Has Python and TensorFlow skills relevant to ML/AI roles. Experience is solid at 4 years, but located in Hyderabad.'), RankedCandidate(candidate_name='Rashmi Kulkarni', rationale='Includes key skills (Python, Flask), has appropriate experience (4 years), but lacks several of the desired advanced skills and located in Pune.'), RankedCandidate(candidate_name='Snehal Patil', rationale='Possesses Python and REST skills, fulfilling part of the requirements. Experience is adequate at 4 years but lacks specific ML/AI focus.'), RankedCandid

In [27]:
len(response.ranked_candidates)

10